In [ ]:
"""
For Google Colab.
"""

# Clone repo
%cd /content
!git clone https://github.com/AHHHHHH0-0/Extending-CoFinDiff.git
%cd Extending-CoFinDiff

# Open terminal and switch branch

## Import


In [ ]:
import sys
import json
import torch
import tqdm
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

sys.path.insert(0, str(Path().resolve().parent.parent))
from denoiser.unet_model_ca_film import UNetDenoiserCAFilm
from diffusion.diffusion_ca_film import DiffusionCAFilm
from preprocessing.condition_encoder import MicroConditionEncoder
from preprocessing.haar_wavelet import HaarWaveletTransform
from config import project_config, diffusion_config, preprocess_config

## Config


In [ ]:
# Seed and device
SEED = project_config.SEED
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    
DEVICE = project_config.DEVICE
CHECKPOINT_PATH = Path('models/ca-film/checkpoints/best_model.pt')

print(f"Device: {DEVICE}")
print(f"Checkpoint: {CHECKPOINT_PATH}")

## Load Models


In [ ]:
# Models
denoiser = UNetDenoiserCAFilm(in_channels=1).to(DEVICE)
micro_encoder = MicroConditionEncoder().to(DEVICE)
diffusion = DiffusionCAFilm()
haar = HaarWaveletTransform()

# Load checkpoint
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
denoiser.load_state_dict(checkpoint['model_state_dict'])
micro_encoder.load_state_dict(checkpoint['micro_encoder_state_dict'])

denoiser.eval()
micro_encoder.eval()

print(f"Loaded denoiser, micro encoder, and diffusion model")

## Set Conditioning


In [ ]:
"""
TODO: Generation configs
"""
N_SAMPLES = 5000
TREND = 0.0
REALIZED_VOL = 50.0
INTEREST_RATE = 5.0
VOLATILITY_INDEX = 20.0

GUIDANCE_SCALE = diffusion_config.GUIDANCE_SCALE

# Build batch tensors
trend_t = torch.full((N_SAMPLES, 1), TREND, dtype=torch.float32, device=DEVICE)
realized_vol_t = torch.full((N_SAMPLES, 1), REALIZED_VOL, dtype=torch.float32, device=DEVICE)
interest_rate_t = torch.full((N_SAMPLES, 1), INTEREST_RATE, dtype=torch.float32, device=DEVICE)
volatility_idx_t = torch.full((N_SAMPLES, 1), VOLATILITY_INDEX, dtype=torch.float32, device=DEVICE)

# Encode micro conditions
with torch.no_grad():
    micro_cond_tokens = micro_encoder(trend_t, realized_vol_t)

# Stack and normalize macro scalars
macro_emb = torch.cat([interest_rate_t, volatility_idx_t], dim=1)
macro_emb = micro_encoder.normalize_macro(macro_emb)

print(f"Generation configs:")
print(f"  N_SAMPLES: {N_SAMPLES}")
print(f"  TREND: {TREND}")
print(f"  REALIZED_VOL: {REALIZED_VOL}")
print(f"  INTEREST_RATE: {INTEREST_RATE}")
print(f"  VOLATILITY_INDEX: {VOLATILITY_INDEX}")
print(f"  GUIDANCE_SCALE: {GUIDANCE_SCALE}")


## Generation


In [ ]:
H, W = preprocess_config.TARGET_SHAPE  # (8, 8)
shape = (N_SAMPLES, 1, H, W)

# Generate samples
samples_2d = diffusion.sample(
    model=denoiser,
    shape=shape,
    micro_cond_tokens=micro_cond_tokens,
    macro_emb=macro_emb,
    guidance_scale=GUIDANCE_SCALE,
)
print(f"Generated {N_SAMPLES} samples shape: {samples_2d.shape}")

# Inverse Haar wavelet: (N_SAMPLES, 1, H, W) -> (N_SAMPLES, T)
samples_1d = haar.inverse(samples_2d.squeeze(1))

# Save generated samples
output_dir = Path('data/generated/ca-film')
output_dir.mkdir(parents=True, exist_ok=True)
generated = {
    'conditions': {
        'trend': TREND,
        'realized_vol': REALIZED_VOL,
        'interest_rate': INTEREST_RATE,
        'volatility_index': VOLATILITY_INDEX,
        'guidance_scale': GUIDANCE_SCALE,
    },
    'samples': samples_1d.cpu().numpy().tolist(),  # list of N_SAMPLES sequences of length T
}

output_path = output_dir / 'generated_returns.json'  # TODO: change
with open(output_path, 'w') as f:
    json.dump(generated, f)

print(f"Generated samples saved to {output_path}")


## Looped Generation for Conditioning Evaluation


In [ ]:
N_SAMPLES = 5000
GUIDANCE_SCALE = diffusion_config.GUIDANCE_SCALE
H, W = preprocess_config.TARGET_SHAPE

TRENDS = [-10, -5, 0, 5, 10]
REALIZED_VOLS = [30, 40, 50, 60, 70]
REGIMES = [
    {'interest_rate': 5.0, 'volatility_index': 20.0},
    {'interest_rate': 8.0, 'volatility_index': 35.0},
]

output_dir = Path('data/generated/ca-film')
output_dir.mkdir(parents=True, exist_ok=True)

shape = (N_SAMPLES, 1, H, W)
total = len(TRENDS) * len(REALIZED_VOLS) * len(REGIMES)
count = 0

for regime in REGIMES:
    for trend in TRENDS:
        for rv in REALIZED_VOLS:
            ir = regime['interest_rate']
            vix = regime['volatility_index']

            trend_t = torch.full((N_SAMPLES, 1), float(trend), dtype=torch.float32, device=DEVICE)
            realized_vol_t = torch.full((N_SAMPLES, 1), float(rv), dtype=torch.float32, device=DEVICE)
            interest_rate_t = torch.full((N_SAMPLES, 1), ir, dtype=torch.float32, device=DEVICE)
            volatility_idx_t = torch.full((N_SAMPLES, 1), vix, dtype=torch.float32, device=DEVICE)

            with torch.no_grad():
                micro_cond_tokens = micro_encoder(trend_t, realized_vol_t)

            macro_emb = torch.cat([interest_rate_t, volatility_idx_t], dim=1)
            macro_emb = micro_encoder.normalize_macro(macro_emb)

            samples_2d = diffusion.sample(
                model=denoiser,
                shape=shape,
                micro_cond_tokens=micro_cond_tokens,
                macro_emb=macro_emb,
                guidance_scale=GUIDANCE_SCALE,
            )
            samples_1d = haar.inverse(samples_2d.squeeze(1))

            filename = f"t{int(trend)}r{int(rv)}i{int(ir)}v{int(vix)}.json"
            generated = {
                'conditions': {
                    'trend': trend,
                    'realized_vol': rv,
                    'interest_rate': ir,
                    'volatility_index': vix,
                    'guidance_scale': GUIDANCE_SCALE,
                },
                'samples': samples_1d.cpu().numpy().tolist(),
            }
            with open(output_dir / filename, 'w') as f:
                json.dump(generated, f)

            count += 1
            print(f"[{count}/{total}] Saved {filename}")

print(f"Done. {count} files saved to {output_dir}")